# Public Opinion EDA — Institutional Trust

Exploratory data analysis of public opinion survey data on institutional trust.

Trust in institutions is one of the most studied dimensions in public opinion research. It shapes political participation, social cohesion, and the legitimacy of democratic systems. This notebook shows how to explore that question systematically: measuring levels, mapping variation across demographic groups, identifying patterns, and communicating findings clearly.

---

## What this notebook covers

| Step | What happens |
|---|---|
| 1 | Setup and configuration |
| 2 | Synthetic dataset generation |
| 3 | Data quality check |
| 4 | Univariate analysis — trust distributions |
| 5 | Institution ranking |
| 6 | Institutional Trust Index (ITI) |
| 7 | Variation by region |
| 8 | Variation by age group |
| 9 | Variation by education |
| 10 | Polarization analysis |
| 11 | Heatmap — subgroup profiles |
| 12 | Export |

**Note:** All data used here is synthetic. No real respondents or client data are included.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Trust variables (1 = no trust at all, 5 = complete trust)
TRUST_VARS = [
    'trust_gov',        # Federal Government
    'trust_judiciary',  # Judiciary
    'trust_congress',   # National Congress
    'trust_military',   # Armed Forces
    'trust_media',      # Media / Press
    'trust_religion',   # Religious Organizations
    'trust_ngo',        # NGOs and Civil Society
    'trust_science',    # Science and Research Institutions
]

INSTITUTION_LABELS = {
    'trust_gov':       'Federal Government',
    'trust_judiciary': 'Judiciary',
    'trust_congress':  'National Congress',
    'trust_military':  'Armed Forces',
    'trust_media':     'Media / Press',
    'trust_religion':  'Religious Orgs',
    'trust_ngo':       'NGOs / Civil Society',
    'trust_science':   'Science Institutions',
}

PALETTE = {
    'main':   '#2E86AB',
    'accent': '#E24B4A',
    'mid':    '#F18F01',
    'soft':   '#B0C4DE',
}

RANDOM_STATE = 42

print('Setup complete.')
print(f'Institutions covered: {len(TRUST_VARS)}')

## 2. Synthetic dataset

We generate a dataset with 1,500 respondents simulating a national probabilistic survey.
Trust scores reflect realistic patterns found in Latin American opinion research:
science and the military tend to score higher, political institutions tend to score lower,
and there is substantial regional and educational variation.

In [ ]:
np.random.seed(RANDOM_STATE)
n = 1500

# Realistic base means per institution (1-5 scale)
base_means = {
    'trust_gov':       2.1,
    'trust_judiciary': 2.4,
    'trust_congress':  1.9,
    'trust_military':  3.3,
    'trust_media':     2.6,
    'trust_religion':  3.0,
    'trust_ngo':       3.2,
    'trust_science':   4.0,
}

data = {}
for var, mean in base_means.items():
    raw = np.random.normal(loc=mean, scale=0.9, size=n)
    data[var] = np.clip(raw, 1, 5).round(0).astype(int)

df = pd.DataFrame(data)

# Demographics
df['age'] = np.random.randint(16, 80, n)
df['age_group'] = pd.cut(
    df['age'], bins=[15, 24, 34, 44, 54, 79],
    labels=['16-24', '25-34', '35-44', '45-54', '55+']
)
df['gender'] = np.random.choice(['Female', 'Male'], n, p=[0.52, 0.48])
df['region'] = np.random.choice(
    ['Southeast', 'Northeast', 'South', 'Center-West', 'North'], n,
    p=[0.42, 0.27, 0.15, 0.08, 0.08]
)
df['education'] = np.random.choice(
    ['Primary', 'Secondary', 'Higher education'], n,
    p=[0.30, 0.45, 0.25]
)

# Regional variation: South and Southeast slightly higher trust in science and NGOs
mask_s = df['region'].isin(['South', 'Southeast'])
df.loc[mask_s, 'trust_science'] = np.clip(df.loc[mask_s, 'trust_science'] + np.random.randint(0, 2, mask_s.sum()), 1, 5)
df.loc[mask_s, 'trust_ngo']     = np.clip(df.loc[mask_s, 'trust_ngo'] + np.random.randint(0, 2, mask_s.sum()), 1, 5)

# Education variation: higher education -> higher trust in science, lower in political institutions
mask_h = df['education'] == 'Higher education'
df.loc[mask_h, 'trust_science']   = np.clip(df.loc[mask_h, 'trust_science'] + 1, 1, 5)
df.loc[mask_h, 'trust_gov']       = np.clip(df.loc[mask_h, 'trust_gov'] - 1, 1, 5)
df.loc[mask_h, 'trust_congress']  = np.clip(df.loc[mask_h, 'trust_congress'] - 1, 1, 5)

print(f'Dataset created: {df.shape[0]} respondents, {df.shape[1]} variables')
df[TRUST_VARS].describe().round(2)

## 3. Data quality check

Before the analysis, we verify that the data is complete and within expected ranges.

In [ ]:
print('=== Missing values ===')
print(df[TRUST_VARS].isnull().sum())

print('\n=== Value ranges (should be 1-5) ===')
for var in TRUST_VARS:
    mn, mx = df[var].min(), df[var].max()
    flag = ' OK' if mn >= 1 and mx <= 5 else ' OUT OF RANGE'
    print(f'  {var}: min={mn}, max={mx}{flag}')

print('\n=== Sample size by group ===')
print('Region:')
print(df['region'].value_counts())
print('\nEducation:')
print(df['education'].value_counts())
print('\nAge group:')
print(df['age_group'].value_counts().sort_index())

## 4. Univariate analysis — trust distributions

We look at the distribution of trust scores for each institution.
The red dashed line marks the mean. The scale runs from 1 (no trust) to 5 (complete trust).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, var in enumerate(TRUST_VARS):
    counts = df[var].value_counts().sort_index()
    axes[i].bar(counts.index, counts.values, color=PALETTE['main'], alpha=0.85, edgecolor='white')
    axes[i].axvline(df[var].mean(), color=PALETTE['accent'], linewidth=1.8, linestyle='--',
                    label=f"Mean: {df[var].mean():.2f}")
    axes[i].set_title(INSTITUTION_LABELS[var], fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Trust score (1-5)', fontsize=8)
    axes[i].set_ylabel('Respondents', fontsize=8)
    axes[i].set_xticks([1, 2, 3, 4, 5])
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution of trust scores by institution', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Institution ranking

Which institutions are trusted most and least?
We rank institutions by mean trust score and visualize the result.

In [ ]:
means = df[TRUST_VARS].mean().rename(INSTITUTION_LABELS).sort_values()
colors = [PALETTE['accent'] if v < 2.5 else PALETTE['mid'] if v < 3.5 else PALETTE['main']
          for v in means.values]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(means.index, means.values, color=colors, edgecolor='white', height=0.6)
ax.axvline(3.0, color='grey', linewidth=1, linestyle='--', alpha=0.6, label='Neutral (3.0)')

for bar, val in zip(bars, means.values):
    ax.text(val + 0.03, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=9)

ax.set_xlim(1, 5.3)
ax.set_xlabel('Mean trust score (1-5)', fontsize=10)
ax.set_title('Institutional trust ranking', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nRanking (descending):')
for inst, score in means.sort_values(ascending=False).items():
    print(f'  {inst:<28} {score:.3f}')

## 6. Institutional Trust Index (ITI)

We build a composite index as the mean of all trust scores per respondent.
This gives a single measure of overall institutional trust that can be used
for subgroup comparisons and modeling.

In [ ]:
df['iti'] = df[TRUST_VARS].mean(axis=1).round(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(df['iti'], bins=30, color=PALETTE['main'], alpha=0.85, edgecolor='white')
axes[0].axvline(df['iti'].mean(), color=PALETTE['accent'], linewidth=2, linestyle='--',
                label=f"Mean ITI: {df['iti'].mean():.3f}")
axes[0].set_xlabel('ITI score')
axes[0].set_ylabel('Respondents')
axes[0].set_title('Distribution of Institutional Trust Index')
axes[0].legend()

# Box plot by education
edu_order = ['Primary', 'Secondary', 'Higher education']
data_box = [df[df['education'] == e]['iti'].values for e in edu_order]
bp = axes[1].boxplot(data_box, labels=edu_order, patch_artist=True,
                     medianprops=dict(color=PALETTE['accent'], linewidth=2))
colors_box = [PALETTE['soft'], PALETTE['mid'], PALETTE['main']]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('ITI score')
axes[1].set_title('ITI distribution by education level')

plt.tight_layout()
plt.show()

print(f'\nITI summary:')
print(f'  Mean:   {df["iti"].mean():.3f}')
print(f'  Median: {df["iti"].median():.3f}')
print(f'  Std:    {df["iti"].std():.3f}')
print(f'  Min:    {df["iti"].min():.3f}')
print(f'  Max:    {df["iti"].max():.3f}')

## 7. Variation by region

Does where people live affect how much they trust institutions?
We compare mean ITI and individual institution scores across Brazil's five regions.

In [ ]:
region_profile = df.groupby('region')['iti'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(region_profile.index, region_profile.values,
              color=PALETTE['main'], alpha=0.85, edgecolor='white')
ax.axhline(df['iti'].mean(), color=PALETTE['accent'], linewidth=1.5, linestyle='--',
           label=f"National mean: {df['iti'].mean():.3f}")

for bar, val in zip(bars, region_profile.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Mean ITI score')
ax.set_title('Institutional Trust Index by region', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print('\nMean ITI by region:')
print(region_profile.round(3).to_string())

## 8. Variation by age group

Trust in institutions often varies across generations.
We compare mean trust scores by age group for each institution.

In [ ]:
age_profile = df.groupby('age_group', observed=True)[TRUST_VARS].mean().round(3)
age_profile.columns = [INSTITUTION_LABELS[v] for v in TRUST_VARS]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(age_profile.index))
width = 0.1

cmap = plt.get_cmap('tab10')
for i, inst in enumerate(age_profile.columns):
    offset = (i - len(age_profile.columns) / 2) * width
    ax.bar(x + offset, age_profile[inst], width=width, label=inst,
           color=cmap(i), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(age_profile.index)
ax.set_ylabel('Mean trust score (1-5)')
ax.set_title('Trust scores by age group and institution', fontsize=12)
ax.legend(fontsize=7, loc='upper right', ncol=2)
ax.set_ylim(1, 5.5)
plt.tight_layout()
plt.show()

print('\nMean trust by age group:')
age_profile

## 9. Variation by education

Education level is one of the strongest predictors of institutional trust patterns.
We compare mean scores per institution across three education levels.

In [ ]:
edu_order = ['Primary', 'Secondary', 'Higher education']
edu_profile = df.groupby('education')[TRUST_VARS].mean().reindex(edu_order).round(3)
edu_profile.columns = [INSTITUTION_LABELS[v] for v in TRUST_VARS]

# Diverging chart: difference between Higher education and Primary
diff = (edu_profile.loc['Higher education'] - edu_profile.loc['Primary']).sort_values()
colors_div = [PALETTE['accent'] if v < 0 else PALETTE['main'] for v in diff.values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar
x = np.arange(len(edu_profile.columns))
w = 0.25
palette_edu = [PALETTE['soft'], PALETTE['mid'], PALETTE['main']]
for i, (edu, color) in enumerate(zip(edu_order, palette_edu)):
    axes[0].bar(x + (i - 1) * w, edu_profile.loc[edu], width=w,
                label=edu, color=color, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(edu_profile.columns, rotation=25, ha='right', fontsize=8)
axes[0].set_ylabel('Mean trust score (1-5)')
axes[0].set_title('Trust scores by education level')
axes[0].legend(fontsize=9)
axes[0].set_ylim(1, 5.5)

# Diverging
axes[1].barh(diff.index, diff.values, color=colors_div, edgecolor='white', height=0.6)
axes[1].axvline(0, color='grey', linewidth=1)
axes[1].set_xlabel('Difference (Higher ed. minus Primary)')
axes[1].set_title('Education gap — Higher education vs Primary')

plt.tight_layout()
plt.show()

print('\nEducation gap (Higher education minus Primary):')
print(diff.round(3).to_string())

## 10. Polarization analysis

Which institutions generate the most disagreement within the population?
High standard deviation indicates that trust is polarized — some people trust
a great deal, others not at all. Low standard deviation indicates broad consensus.

In [ ]:
polarization = df[TRUST_VARS].std().rename(INSTITUTION_LABELS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors_pol = [PALETTE['accent'] if v > polarization.median() else PALETTE['main']
              for v in polarization.values]
bars = ax.barh(polarization.index, polarization.values,
               color=colors_pol, edgecolor='white', height=0.6)
ax.axvline(polarization.median(), color='grey', linewidth=1.2, linestyle='--',
           label=f'Median std: {polarization.median():.3f}')

for bar, val in zip(bars, polarization.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Standard deviation of trust score')
ax.set_title('Institutional trust polarization (std dev)', fontsize=12)
ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nPolarization ranking (highest to lowest):')
print(polarization.round(3).to_string())

## 11. Heatmap — subgroup profiles

A heatmap gives a compact overview of how trust varies across all institutions
and all regional subgroups simultaneously. Darker blue = higher trust.

In [ ]:
heatmap_data = df.groupby('region')[TRUST_VARS].mean().round(2)
heatmap_data.columns = [INSTITUTION_LABELS[v] for v in TRUST_VARS]

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(heatmap_data.values, cmap='Blues', aspect='auto', vmin=1, vmax=5)
plt.colorbar(im, ax=ax, label='Mean trust score (1-5)')

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=25, ha='right', fontsize=9)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=9)

for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.iloc[i, j]
        text_color = 'white' if val > 3.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=text_color)

ax.set_title('Mean trust scores by region and institution', fontsize=12)
plt.tight_layout()
plt.show()

## 12. Export

In [ ]:
export_cols = ['age', 'age_group', 'gender', 'region', 'education', 'iti'] + TRUST_VARS
df[export_cols].to_csv('institutional_trust_data.csv', index=False)

summary = df[TRUST_VARS].agg(['mean', 'median', 'std']).round(3)
summary.columns = [INSTITUTION_LABELS[v] for v in TRUST_VARS]
summary.to_csv('trust_summary.csv')

print('Files saved:')
print('  institutional_trust_data.csv -- full dataset with ITI')
print('  trust_summary.csv            -- mean, median and std per institution')
print(f'\nFinal dataset: {df.shape[0]} respondents')
print(f'National mean ITI: {df["iti"].mean():.3f}')
print(f'\nTop 3 most trusted institutions:')
top3 = df[TRUST_VARS].mean().rename(INSTITUTION_LABELS).sort_values(ascending=False).head(3)
for inst, score in top3.items():
    print(f'  {inst}: {score:.3f}')
print(f'\nBottom 3 least trusted institutions:')
bot3 = df[TRUST_VARS].mean().rename(INSTITUTION_LABELS).sort_values().head(3)
for inst, score in bot3.items():
    print(f'  {inst}: {score:.3f}')